# Solving CSPs (Constraint Satisfaction Problems): backtracking and consistency

_Artificial Intelligence (CS221)_

**Try a value, check the rules, and back up the moment you get stuck.**

To solve a CSP, assign variables one at a time. This is backtracking search.

     After each choice, check the rules. If a rule is already broken, undo and try another value.

---

This notebook builds **backtracking search** from scratch, one piece at a time. We will color a map of Australia so that no two neighboring states share a color, choosing one state at a time and undoing any choice that breaks a border rule. Run each cell, read the note above it, then tackle the practice problems at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

We'll solve the **Australia map-coloring** CSP: assign one of three colors to each state so that no two states sharing a border get the same color. We build it in three steps: (1) the variables, colors, and border constraints, (2) a consistency check, (3) the recursive backtracking search.

### Step 1 — Define the variables, colors, and constraints

The **variables** are the seven states. Each variable's **domain** is the same three colors. The **constraints** are the borders: each `edge` is a pair of states that may not share a color. Note `T` (Tasmania) appears in no edge — it is an island, so it can take any color.

In [ ]:
variables = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
colors = ["red", "green", "blue"]

# Each edge is a pair of neighboring states that must differ in color.
edges = [
    ("WA", "NT"), ("WA", "SA"), ("NT", "SA"), ("NT", "Q"),
    ("SA", "Q"), ("SA", "NSW"), ("SA", "V"), ("Q", "NSW"), ("NSW", "V"),
]

### Step 2 — Check whether a color is consistent

Before committing a color to a state, we ask: does it clash with any *already-assigned* neighbor? We scan every border that touches this state and reject the color if the neighbor on the other side already has it. Neighbors that are not yet assigned cannot clash, so we skip them.

In [ ]:
def ok(var, color, assign):
    for a, b in edges:                  # check every border that touches `var`
        if a == var and b in assign and assign[b] == color:
            return False
        if b == var and a in assign and assign[a] == color:
            return False
    return True

### Step 3 — Backtracking search

We assign states one at a time. For the next unassigned state we try each color in turn. If a color is consistent, we tentatively commit it and **recurse** on the rest. If the recursion succeeds we return the full solution; if it fails we **undo** the assignment (`del assign[var]`) and try the next color — this is the *backtracking* step. Running out of colors returns `None`, signaling failure to the caller.

In [ ]:
def backtrack(assign):
    if len(assign) == len(variables):
        return assign                   # all variables assigned: done

    # Pick the next variable that has no color yet.
    var = next(v for v in variables if v not in assign)

    for color in colors:
        if ok(var, color, assign):
            assign[var] = color         # tentatively commit this color
            result = backtrack(assign)  # recurse on the remaining variables
            if result is not None:
                return result
            del assign[var]             # undo and try the next color
    return None                         # no color worked: signal failure

solution = backtrack({})
print("solution found:")
for v in variables:
    print(" ", v, "=", solution[v])

## Visualize the data & results

_What valid 3-coloring does backtracking find for the seven states of Australia?_

We'll re-run the same search and then chart the color each state received. We do it in three steps: (1) restate the CSP and search, (2) run it, (3) draw the bar chart.

### Step 1 — Restate the CSP and the search

This is the same problem and the same algorithm as above, kept self-contained so the plot cell can run on its own. The helper is named `consistent` here and uses `assign.get(b)` to fetch a neighbor's color (returning `None` if unassigned).

In [ ]:
states = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
colors = ["red", "green", "blue"]
borders = [
    ("WA", "NT"), ("WA", "SA"), ("NT", "SA"), ("NT", "Q"),
    ("SA", "Q"), ("SA", "NSW"), ("SA", "V"), ("Q", "NSW"), ("NSW", "V"),
]

def consistent(var, col, assign):
    for a, b in borders:
        if a == var and assign.get(b) == col:
            return False
        if b == var and assign.get(a) == col:
            return False
    return True

def backtrack(assign):
    if len(assign) == len(states):
        return assign
    var = next(v for v in states if v not in assign)
    for col in colors:
        if consistent(var, col, assign):
            assign[var] = col
            if backtrack(assign):
                return assign
            del assign[var]
    return None

### Step 2 — Run the search and encode the result

We run `backtrack` to get the solution, then map each color name to a numeric code (so it can be a bar height) and to a hex color (so the bar is drawn in that color).

In [ ]:
sol = backtrack({})

code_of = {"red": 1, "green": 2, "blue": 3}
palette = {"red": "#ff7b72", "green": "#7ee787", "blue": "#4ea1ff"}

vals = [code_of[sol[s]] for s in states]
cols = [palette[sol[s]] for s in states]

### Step 3 — Chart the coloring

Each bar is one state, drawn in its assigned color and labeled with the color name. Because adjacent states differ, neighboring bars never share a color.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(states, vals, color=cols)

# Label each bar with the color name it was assigned.
for i, s in enumerate(states):
    ax.text(i, vals[i], sol[s], ha="center", va="bottom")

ax.set_title("Australia 3-coloring found by backtracking")
plt.show()